```{contents}
```

## The Transformer

The **Transformer** is a groundbreaking neural network architecture for sequence transduction tasks (e.g., machine translation, summarization, and language modeling).
It eliminates the use of **recurrence** (RNNs) and **convolution** (CNNs), relying **solely on attention mechanisms**.
This design yields models that are **faster, more parallelizable, and higher-performing** than previous state-of-the-art systems.

---

### Architecture

The Transformer adopts a standard **encoder–decoder framework**.

1. **Encoder**
   The encoder transforms an input sequence of tokens ($x_1, \dots, x_n$) into a continuous representation sequence
   $$
   z = (z_1, \dots, z_n)
   $$
   It consists of **six identical layers**, each containing:

   * Multi-Head Self-Attention
   * Position-wise Feed-Forward Network (FFN)

2. **Decoder**
   The decoder generates the target sequence ($y_1, \dots, y_m$) **auto-regressively**, consuming previously generated outputs to predict the next token.
   It also consists of **six identical layers**, each with:

   * Masked Multi-Head Self-Attention
   * Encoder–Decoder Attention
   * Feed-Forward Network

All sub-layers use **residual connections** and **layer normalization**.
Every layer and embedding produces outputs of dimension

$$
d_{\text{model}} = 512
$$

---

### **II. Core Components of Transformer Layers**

#### **Multi-Head Attention**

The Transformer’s foundation is the **attention mechanism**, which maps a **query** $Q$, a set of **keys** $K$, and **values** $V$ to an output:

$$
\text{Attention}(Q, K, V) = \text{softmax}!\left(\frac{QK^T}{\sqrt{d_k}}\right)V
$$
This computes a **weighted sum of values**, where the weights represent the relevance of each key to the query.

##### **Multi-Head Attention Process**

Instead of a single attention operation, the Transformer uses **multiple heads**:

1. **Projection:** $Q, K, V$ are linearly projected $h$ times using learned matrices into lower-dimensional subspaces ($d_k = d_v = d_{\text{model}} / h$).
2. **Parallel Attention:** Each head performs Scaled Dot-Product Attention independently.
3. **Concatenation:** Outputs from all heads are concatenated and linearly projected to form the final result.

In the base Transformer:
$$
h = 8,\quad d_k = d_v = 64
$$

This enables the model to **attend to different information types** simultaneously (syntax, semantics, positional relations, etc.).

#### **Attention Usage Across Layers**

| **Location**                | **Type**                  | **Inputs**                                | **Purpose**                                                       |
| --------------------------- | ------------------------- | ----------------------------------------- | ----------------------------------------------------------------- |
| **Encoder**                 | Self-Attention            | $Q, K, V$ from encoder outputs          | Enables each token to attend to all others in the input sequence. |
| **Decoder (1st sub-layer)** | Masked Self-Attention     | $Q, K, V$ from previous decoder layer   | Prevents attending to future tokens (auto-regression).            |
| **Decoder (2nd sub-layer)** | Encoder–Decoder Attention | $Q$ from decoder, $K, V$ from encoder | Allows each decoder token to focus on relevant encoder positions. |

---

#### **Position-Wise Feed-Forward Networks (FFN)**

Each layer includes a fully connected feed-forward network applied independently to each position:

$$
\text{FFN}(x) = \max(0, xW_1 + b_1)W_2 + b_2
$$
with:

* $d_{\text{model}} = 512$
* $d_{\text{ff}} = 2048$

This sub-layer increases representational capacity and introduces non-linearity.

---

### **III. Input Representation and Positional Encoding**

Because the Transformer lacks recurrence or convolution, it must explicitly encode token order.

1. **Embeddings:**
   Token indices are converted to learned embedding vectors of size $d_{\text{model}}$.

2. **Positional Encoding (PE):**
   Positional encodings are added to embeddings to encode sequence order using sinusoidal functions:
   $$
   PE_{(pos, 2i)} = \sin\left(\frac{pos}{10000^{2i / d_{\text{model}}}}\right), \quad
   PE_{(pos, 2i+1)} = \cos\left(\frac{pos}{10000^{2i / d_{\text{model}}}}\right)
   $$
   This allows the model to attend by **relative position** and **extrapolate** to longer sequences.

---

### **IV. Advantages over Recurrent and Convolutional Models**

#### **1. Parallelization**

RNNs process tokens sequentially $(O(n)) steps$, while self-attention connects all tokens in **constant depth** ($O(1)$).
This yields **massive parallel training efficiency**.

#### **2. Shorter Dependency Paths**

| Model          | Maximum Path Length | Complexity   |
| -------------- | ------------------- | ------------ |
| Self-Attention | $O(1)$              | $O(n^2 d)$   |
| RNN            | $O(n)$              | $O(n d^2)$   |
| CNN (ConvS2S)  | $O(\log_k n)$       | $O(k n d^2)$ |

Self-attention provides **direct global connections**, improving long-range dependency modeling.

#### **3. Performance**

The Transformer achieved **state-of-the-art BLEU scores**:

* **EN → DE:** 28.4
* **EN → FR:** 41.8
  Both outperforming previous RNN/CNN ensembles by over 2 BLEU points.
  The base model trained in **≈12 hours on 8×P100 GPUs**.

#### **4. Interpretability**

Attention weights are human-interpretable — visualizing them reveals syntactic and semantic alignments (e.g., subject–verb dependencies, pronoun references).

---

### **V. Conceptual Analogy**

Imagine a **research assistant** searching through thousands of documents:

* The **query (Q)** is your current question.
* The **keys (K)** represent the topic summaries of all documents.
* The **values (V)** are the document contents.

Instead of reading sequentially, the assistant compares the query with all keys simultaneously, ranks relevance (via attention weights), and blends the relevant content (values).
This mechanism enables **instant, parallel context retrieval**, replacing the need for step-by-step scanning (as in RNNs).

---

**Summary**

| **Component**                | **Purpose**                   | **Key Equation**                             |
| ---------------------------- | ----------------------------- | -------------------------------------------- |
| Scaled Dot-Product Attention | Context weighting             | $\text{softmax}(\frac{QK^T}{\sqrt{d_k}})V$ |
| Multi-Head Attention         | Multiple contextual subspaces | Concatenate parallel heads                   |
| Feed-Forward Network         | Non-linear transformation     | $\text{ReLU}(xW_1 + b_1)W_2 + b_2$         |
| Positional Encoding          | Sequence order encoding       | Sine–Cosine positional patterns              |
| Residual + LayerNorm         | Stable training               | $\text{LayerNorm}(x + \text{Sublayer}(x))$ |

---

**In Essence**

> The **Transformer** functions as a *parallel global reasoning system*.
> It replaces sequential computation with **attention-based relational mapping**, enabling faster training, longer dependency learning, and more interpretable behavior — the foundation for modern large language models like **BERT**, **GPT**, and **T5**.


```{dropdown} Click here for Sections
```{tableofcontents}